# Week 4：PyTorch 基础 + 表格 MLP

目标：
1. 理解 tensor、autograd、nn.Module、Dataset/DataLoader、训练循环、早停；
2. 做一个表格 MLP，预测 Response；
3. 训练集再切验证集做早停，测试集只评估一次；
4. 与第 2 周的 RF/Logistic 基线比较；
5. 结果追加到 results/baseline_metrics.csv。

注：运行前需先安装pyTorch（只需一次）
anaconda powershell prompt运行
cd “你的路径”
conda activate ctlearn
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [1]:
# 导入依赖，固定随机种子
from pathlib import Path
import random
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 预处理与评价
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    confusion_matrix,
    roc_curve,
)
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt

# 固定随机种子，尽量让每次运行结果一致
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 有 GPU 就用 GPU，没有就用 CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("torch:", torch.__version__)
print("device:", DEVICE)

torch: 2.15.0+cpu
device: cpu


In [3]:
# Pytorch基础自测
# tensor + autograd 的最小例子
# 这是一段最小可运行的自动微分(autograd)示例，对应了神经网络训练的核心步骤：loss.backward()如何计算梯度

#创建需要求导的张量，resquires_grad = True是指：请跟踪对x的所有操作，之后要对它求梯度
#此时x是一个叶子张量(leaf tensor)，且：
#x.requires_grad   # True   一般默认False，只有手动设为True的张量才会被求导
#x.grad            # None（还没反向传播）
#x.is_leaf         # True
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

#前向求导
#逐元素平方再求和
#由于x.requires_grad=True，因此PyTorch会动态记录这些运算，构建计算图
#x  ──►  x²  ──►  sum  ──►  y
#     (PowBackward)  (SumBackward)
#y不是用户直接创建的叶子，而是运算结果，故：
#y.requires_grad   # True（继承自 x）
#y.grad_fn         # <SumBackward0>（记录了它是怎么算出来的）
#y.is_leaf         # False
y = (x ** 2).sum()

#反向传播
#这是反向模式自动微分(reverse-mode autodiff)，即反向传播
#沿着计算图从y往回走，使用链式法则计算dy/dx，推导出dy/dxi=2xi
#代入得到x=[1,2,3]，dy/dx=[2,4,6]，梯度被写入x.grad
#注：y为标量故可以直接调用y.backward，如果为向量，则需要传入一个与y同形状的gradient参数，指定每个输出分量的权重
y.backward()

print("x:", x)
print("y:", y.item())
print("dy/dx:", x.grad)   # 应该是 2x = [2, 4, 6]

x: tensor([1., 2., 3.], requires_grad=True)
y: 14.0
dy/dx: tensor([2., 4., 6.])


# 1. 叶子张量(leaf tensor)
是指用户自行创建的，并非中间运算产生的张量
叶子节点                中间节点                 根节点
   x        ──►       x²        ──►        sum        ──►  y
（你创建的）          （运算结果）          （运算结果）
PyTorch默认只把梯度保存在叶子张量的 .grad中
# 2. 链式法则
dy/dx = （dy/du） · （du/dx）
多路径时：梯度相加，即一个变量在计算图中被用多次时(出现在两条路径上)，反向传播时，所有路径的梯度在叶子处相加
因此由于链式法则，每一层只需要知道自己的局部导数，将上游传下来的梯度×局部导数，再传给下游，即可实现自动微分
# 3. x.grad
是张量的一个属性，存放目标函数对x的梯度，即保存dy/dx
注意：x.grad存放梯度是累加的，而非覆盖的
如上述代码，如果计算2次y.backward()，得到的结果会是[4,8,12]
因此训练循环时必须清零：optimizer.zero_grad()

In [4]:
# 读取数据、切分训练/验证/测试
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(ROOT / "data/raw/chowell_train.csv")
test = pd.read_csv(ROOT / "data/raw/chowell_test.csv")

TARGET = "Response (1:Responder; 0:Non-responder)"

NUMERIC_FEATURES = [
    "Age",
    "Sex (1:Male; 0:Female)",
    "BMI",
    "Stage (1:IV; 0:I-III)",
    "NLR",
    "Platelets",
    "HGB",
    "Albumin",
    "Drug (1:Combo; 0:PD1/PDL1orCTLA4)",
    "TMB",
    "FCNA",
    "HED",
    "HLA_LOH",
    "MSI (1:Unstable; 0:Stable_Indeterminate)",
    "MSI_SCORE",
    "Chemo_before_IO (1:Yes; 0:No)",
]

CATEGORICAL_FEATURES = ["Cancer_type_grouped_2"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_all_train = train[FEATURES].copy()
y_all_train = train[TARGET].astype(int).values

X_test = test[FEATURES].copy()
y_test = test[TARGET].astype(int).values

# 从训练集里再切 20% 做验证集，用于早停；
# 测试集完全不参与训练和早停。
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all_train,
    y_all_train,
    test_size=0.2,
    stratify=y_all_train,
    random_state=SEED,
)

print("train:", X_tr.shape, "val:", X_val.shape, "test:", X_test.shape)
print("positive rate — train: %.4f, val: %.4f, test: %.4f" % (
    y_tr.mean(), y_val.mean(), y_test.mean()
))

train: (947, 17) val: (237, 17) test: (295, 17)
positive rate — train: 0.2693, val: 0.2700, test: 0.3051


In [5]:
# 预处理，仅在训练集fit
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), #模型学习中位数
    ("scaler", StandardScaler()),#模型学习均值和标准差
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),#学习众数
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),#学习训练集中的类别列表和顺序
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
], remainder="drop")

# 只在训练集上 fit，验证集/测试集只 transform
X_tr_np = preprocessor.fit_transform(X_tr).astype(np.float32)
X_val_np = preprocessor.transform(X_val).astype(np.float32)
X_test_np = preprocessor.transform(X_test).astype(np.float32)

print("input dim:", X_tr_np.shape[1])

input dim: 32


fit：是指模型学习数据
transform：是指用模型学到的数据再回过头去处理数据
例如：
X_tr_np = preprocessor.fit_transform(X_tr).astype(np.float32)
是指再训练集学习到中位数、均值、标准差、类别列表后，再用学到的数据去处理训练集中的缺失值

X_val_np = preprocessor.transform(X_val).astype(np.float32)
X_test_np = preprocessor.transform(X_test).astype(np.float32)
由于要防止数据泄露，故对验证集和测试集不进行fit学习(不看它们的数据),而是直接用再训练集学习到的值去处理这两个数据集中的缺失值

In [6]:
#构建Dataset和DataLoader

#本部分目的是将Numpy数组包装为PyTorch可以高效迭代的Dataset+DataLoader
#训练时每次取一小批(64条)样本做前向、反向、更新，而非一次性喂全部数据

#每个mini-batch包含64个样本
#一个epoch(完整遍历一次训练集)会被切成若干个batch
BATCH_SIZE = 64
#使用mini-batch的原因：
#    比全量梯度下降更新的更频繁、收敛更快
#    比单样本SGD更稳定，能利用矩阵运算加速
#    64/128/256是常用取值，受内存和样本的影响


# TensorDataset 是最简单的 Dataset：把特征和标签一一对应
#训练集
#将特征张量和标签张量按照"第一维对齐"打包，即第i个样本为(特征[i]，标签[i])，这要求两个张量第一个维度(947)相同
train_ds = TensorDataset(
     #将Numpy特征矩阵(947, 17)转成PyTorch张量
    torch.tensor(X_tr_np, dtype=torch.float32),
    #将Numpy标签矩阵(947,)转成PyTorch张量
    #使用.view(-1,1)把形状(947,)变成(947, 1)，-1表示这一维自动推断(此处推断为947)，
    #此处目的是使标签保持同形状，防止loss计算出问题
    torch.tensor(y_tr, dtype=torch.float32).view(-1, 1),
)

#验证集/测试集
val_ds = TensorDataset(
    torch.tensor(X_val_np, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32).view(-1, 1),
)
test_ds = TensorDataset(
    torch.tensor(X_test_np, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32).view(-1, 1),
)

#注：用BCEWithLogitsLoss / BCELoss计算loss时需要view(1,-1)，防止形状不匹配导致报错
#    如果用CrossEntropyLoss损失函数，需要dtype=torch.long, 形状(batch,)，无需view(-1,1)


#按照批次迭代构建DataLoader
# 训练集 shuffle=True；验证/测试集不 shuffle
# shuffle=True的原因：如果数据按照类别/时间排序，不打乱会让每个batch的分布偏倚，打乱以后更接近整体分布，梯度估计更稳，也可以避免模型学到"样本顺序"这种虚假规律
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True, #每个epoch重新打乱样本顺序
    generator=torch.Generator().manual_seed(SEED), #固定打乱的随机种子，保证可复现
)
#验证集/测试集的评估指标与顺序无关，打乱没有收益；且固定顺序便于把预测结果和原始标签逐条对齐；减少不确定性，使结果可复现
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

#len(loader)返回batch的数量
print("batches — train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))

batches — train: 15 val: 4 test: 5


In [ ]:
#定义MLP模型、损失函数、优化器，详细内容见学习笔记相关部分
#这一部分定义了一个用于表格数据的3层MLP，并配置好：
#输入(32维)→64→32→1(logit)
#损失函数BCEWithLogitsLoss + pos_weight（处理 27% 的不平衡）
#优化器：Adam(lr=1e-3, weight_decay=1e-4)

#总体任务：配置好小型MLP，输出原始logit，再用pos_weight≈2.712的BCEWithLogitsloss缓解阳性只占27%的不平衡，最后用Adam优化

#继承nn.Module
#nn.Modle是所有神经网络模块的基类，继承后自动获得：
#   参数注册与管理(model.parameters())
#   前向调用model(x)的能力
#   train()/eval()模式切换
#   state_dirc()保存/加载权重
#   .to(device)搬移参数
class TabularMLP(nn.Module):
    """
    简单的表格 MLP：
    输入 -> 64 -> 32 -> 1
    隐藏层用 ReLU + Dropout；最后一层输出 logit，不加 sigmoid。
    """

    #初始化父类，否则PyTorch无法注册子模块
    #参数含义：
    #    input_dim是输入特征数，即之前prep中fit_transform的X_tr_np，预处理后是32
    #    hidden1=64，第一隐藏层的宽度
    #    hidden2=32，第二隐藏层的宽度
    #    dropout=0.3，每个Dropout层丢弃30%神经元
    def __init__(self, input_dim, hidden1=64, hidden2=32, dropout=0.3):
        super().__init__()

        #nn.Sequential堆叠网络，是一个"容器"，按照顺序执行里面的层，输入的x会依次流过每一层
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1), #形状(B,32)(B,64)，全连接y=x·W^T+b
            nn.ReLU(), #形状不变，非线性max(0,x)，没有ReLU，那么两层Linear叠加等价于一层Linear，网络会退化为线性模型，ReLU引入非线性，使MLP能拟合复杂关系
            nn.Dropout(dropout),#形状不变，训练时随机把30%的元素置0，防止过拟合；注意model.eval()会不丢弃而原样通过，故训练和评估前需正确切换model.train()/model.eval()
            nn.Linear(hidden1, hidden2),#形状变为(B,64)(B,32)，是第二个隐藏层
            nn.ReLU(),#形状不变，非线性
            nn.Dropout(dropout),#同上
            nn.Linear(hidden2, 1),#形状变为(B,32)(B,1)，输出1个logit(未归一化的分数，二分类只需1个输出)，配合BCEWithLogitLoss使用(该损失函数内部自带sigmoid，且数值更稳定)
        )

    #定义数据如何流经网络
    #调用时用model(x)，而非model.forward(x)——__call__会额外处理hooks等机制
    #输入x形状(B,32),输出(B,1)
    def forward(self, x):
        return self.net(x)

#实例化模型
#X_tr_np是预处理后的矩阵，本数据为16个数值特征+16个癌种的one-hot=32维
#.to(DIVICE)把模型所有参数搬到DEVICE(本机器为CPU)，如果有GPU就搬到显存
#注意输入数据也要.to(DEVICE)，否则会报"device mismatch"
model = TabularMLP(input_dim=X_tr_np.shape[1]).to(DEVICE)
print(model)


# 阳性只占约 27%，用 pos_weight 让损失函数更关注阳性
# 使模型更愿意预测阳性，提高Recall，但是Precision可能下降
pos_weight = torch.tensor(
    [(y_tr == 0).sum() / (y_tr == 1).sum()],
    dtype=torch.float32,
    device=DEVICE,
)

# 损失函数BCEWithLogitsLoss 内部包含 sigmoid，输入必须是 logit
# target必须是float，形状(B,1)，与view(-1,1)的处理相对应
# 输出logit的形状，(B,1)，与target对齐
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


#配置优化器Adam
optimizer = torch.optim.Adam(
    model.parameters(), #所有需要梯度的叶子张量(权重、偏置)
    lr=1e-3, #学习率0.001，Adam的常用默认值
    weight_decay=1e-4, #L2正则，一致权重过大，防止过拟合，此处使耦合式L2，更现代的做法是AdamW
)

print("pos_weight:", float(pos_weight.item()))